# Credit Factor Hierarchy

**Purpose:** End-to-end walkthrough of the credit factor hierarchy feature.

**Prerequisites:** Familiarity with bonds, P&L attribution, and spread curves.

**In this notebook:**
1. Build synthetic spread history for 6 issuers (24 months).
2. Define issuer tags and a two-level (Rating, Region) hierarchy.
3. Calibrate a `CreditFactorModel`.
4. Save the artifact to JSON; reload it.
5. Decompose a two-date period with `decompose_levels` and `decompose_period`.
6. Run attribution on a small synthetic portfolio.
7. Build a credit vol forecast report.

In [ ]:
import json
import math
import tempfile
from datetime import date as dt_date, timedelta
from pathlib import Path

import numpy as np

from finstack.valuations import (
    CreditCalibrator,
    CreditFactorModel,
    FactorCovarianceForecast,
    attribute_pnl_from_spec,
    decompose_levels,
    decompose_period,
)

print("Imports OK.")

## Step 1 — Synthetic spread history

We generate 24 months of spread data for 6 issuers: 3 IG (EU, NA, APAC) and
3 HY (EU, NA, APAC).  Each issuer's spread is a linear function of a generic
factor plus a small idiosyncratic noise term.  All random numbers use fixed
seeds to ensure deterministic CI execution.

In [ ]:
rng = np.random.default_rng(42)  # fixed seed — deterministic

N_MONTHS = 24
AS_OF = "2024-03-31"

# Monthly dates ending at AS_OF
end_date = dt_date(2024, 3, 31)
dates = []
d = end_date
for _ in range(N_MONTHS):
    dates.append(d.isoformat())
    d = d - timedelta(days=30)
dates.reverse()

# Generic factor: CDX IG 5Y proxy, mean 100 bp with small trend
generic_values = [100.0 + 0.5 * math.sin(i * 0.4) for i in range(N_MONTHS)]

ISSUERS = [
    ("ISSUER-A", "IG", "EU"),
    ("ISSUER-B", "IG", "NA"),
    ("ISSUER-C", "IG", "APAC"),
    ("ISSUER-D", "HY", "EU"),
    ("ISSUER-E", "HY", "NA"),
    ("ISSUER-F", "HY", "APAC"),
]

spreads: dict[str, list] = {}
tags: dict[str, dict] = {}
asof_spreads: dict[str, float] = {}

for k, (issuer_id, rating, region) in enumerate(ISSUERS):
    base = 80.0 + k * 30.0  # IG: ~80-170 bp, HY: ~170-230 bp
    beta_pc = 0.6 + 0.05 * k
    noise = rng.normal(0, 1.5, size=N_MONTHS)
    series = [
        base + beta_pc * (generic_values[i] - 100.0) + noise[i]
        for i in range(N_MONTHS)
    ]
    spreads[issuer_id] = series
    tags[issuer_id] = {"rating": rating, "region": region}
    asof_spreads[issuer_id] = series[-1]

print(f"Dates: {dates[0]} .. {dates[-1]}  ({len(dates)} months)")
print(f"Issuers: {list(spreads.keys())}")
print("As-of spreads (bp):")
for k, v in asof_spreads.items():
    print(f"  {k}: {v:.1f}")

## Step 2 — Hierarchy spec and calibration config

In [ ]:
calibration_config = {
    "policy": "globally_off",
    "hierarchy": {"levels": ["rating", "region"]},
    "min_bucket_size_per_level": {"per_level": [2, 2]},
    "vol_model": "sample",
    "covariance_strategy": "diagonal",
    "beta_shrinkage": "none",
    "use_returns_or_levels": "returns",
    "annualization_factor": 12.0,
}

calibration_inputs = {
    "history_panel": {"dates": dates, "spreads": spreads},
    "issuer_tags": {"tags": tags},
    "generic_factor": {
        "spec": {"name": "CDX IG 5Y", "series_id": "cdx.ig.5y"},
        "values": generic_values,
    },
    "as_of": AS_OF,
    "asof_spreads": asof_spreads,
    "idiosyncratic_overrides": {},
}

print("Config hierarchy levels:", calibration_config["hierarchy"]["levels"])
print("Input issuers:", list(calibration_inputs["issuer_tags"]["tags"].keys()))

## Step 3 — Calibrate `CreditFactorModel`

In [ ]:
calibrator = CreditCalibrator(json.dumps(calibration_config))
model = calibrator.calibrate(json.dumps(calibration_inputs))

print(f"schema_version : {model.schema_version}")
print(f"as_of          : {model.as_of}")
print(f"n_levels       : {model.n_levels}")
print(f"n_issuers      : {model.n_issuers}")
print(f"n_factors      : {model.n_factors}")
print(f"level_names    : {model.level_names()}")
print(f"issuer_ids     : {model.issuer_ids()}")
print(f"factor_ids     : {model.factor_ids()}")

## Step 4 — Save and reload artifact

In [ ]:
# Save to a temporary file and reload — verifies the JSON round-trip.
with tempfile.NamedTemporaryFile(mode="w", suffix=".json", delete=False) as f:
    artifact_path = Path(f.name)
    f.write(model.to_json())

artifact_json = artifact_path.read_text()
model_reloaded = CreditFactorModel.from_json(artifact_json)

assert model_reloaded.schema_version == model.schema_version
assert model_reloaded.n_issuers == model.n_issuers
assert model_reloaded.n_factors == model.n_factors
assert model_reloaded.issuer_ids() == model.issuer_ids()

# Clean up
artifact_path.unlink()

print("Artifact saved and reloaded successfully.")
print(f"Reloaded: {model_reloaded}")

## Step 5 — Decompose a two-date period

`decompose_levels` computes a snapshot: generic factor value, per-level bucket
values, and per-issuer residual adders.  `decompose_period` diffs two snapshots.

In [ ]:
# T0: month 22 (Feb 2024 proxy)
T0_DATE = "2024-02-29"
T0_GENERIC = 99.5

# T1: as-of (March 2024)
T1_DATE = AS_OF
T1_GENERIC = 101.2

spreads_json = json.dumps(asof_spreads)  # same observed spreads at both dates

snap_t0 = decompose_levels(model, spreads_json, T0_GENERIC, T0_DATE)
snap_t1 = decompose_levels(model, spreads_json, T1_GENERIC, T1_DATE)

period = decompose_period(snap_t0, snap_t1)

print(f"Period: {period.from_date} → {period.to_date}")
print(f"Δ generic (bp): {period.d_generic:.4f}")

for lvl in range(period.n_levels):
    deltas = period.level_deltas(lvl)
    level_name = model.level_names()[lvl]
    print(f"Level {lvl} ({level_name}) Δ bucket values:")
    for bucket, delta in sorted(deltas.items()):
        print(f"  {bucket}: {delta:+.4f} bp")

d_adder = period.d_adder()
print("Δ adder per issuer (bp):")
for issuer, delta in sorted(d_adder.items()):
    print(f"  {issuer}: {delta:+.4f}")

## Step 6 — Attribution on a small portfolio

We run `attribute_pnl_from_spec` with an `AttributionEnvelope` JSON that
embeds the model inline.  The market context has a flat discount curve and no
hazard curves (so credit P&L is zero in this synthetic example — the purpose
here is to confirm the spec executes end-to-end without errors).

In [ ]:
def flat_discount_curve(base_date: str, rate: float) -> dict:
    """Minimal MarketContextState dict with one flat discount curve.

    CurveState uses #[serde(tag = "type", rename_all = "snake_case")] so the
    discount variant is serialised as {"type": "discount", ...} (internally
    tagged, not externally tagged).  DiscountCurve serialises via
    RawDiscountCurve which has fields: id, base, day_count, knot_points
    (List[Tuple[float, float]]), interp_style, extrapolation.
    """
    # Build knot_points as (year_fraction, discount_factor) pairs.
    tenors = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 30.0]
    knot_points = [[t, math.exp(-rate * t)] for t in tenors]

    return {
        "version": 2,  # MARKET_CONTEXT_STATE_VERSION; must match finstack-core/src/market_data/context/state_serde.rs
        "curves": [
            {
                "type": "discount",    # snake_case tag for CurveState::Discount
                "id": "USD-OIS",
                "base": base_date,
                "day_count": "Act365F",  # DayCount uses PascalCase serde
                "knot_points": knot_points,
                "interp_style": "log_linear",
                "extrapolation": "flat_forward",
            }
        ],
        "fx": None,
        "surfaces": [],
        "prices": {},
        "series": [],
        "inflation_indices": [],
        "dividends": [],
        "credit_indices": [],
        "collateral": {},
        "fx_delta_vol_surfaces": [],
        "hierarchy": None,
        "vol_cubes": [],
    }


AS_OF_T0 = "2025-01-15"
AS_OF_T1 = "2025-01-16"
market_t0 = flat_discount_curve(AS_OF_T0, 0.04)
market_t1 = flat_discount_curve(AS_OF_T1, 0.04 + 0.0001)  # 1 bp shift

print("Market states built.")

In [ ]:
def make_bond_spec(bond_id: str, coupon: float = 0.05, years: int = 5) -> dict:
    """Minimal bond instrument spec for attribution."""
    return {
        "type": "bond",
        "spec": {
            "id": bond_id,
            "notional": {"amount": 1_000_000.0, "currency": "USD"},
            "issue_date": "2025-01-01",
            "maturity": f"{2025 + years}-01-01",
            "cashflow_spec": {
                "Fixed": {
                    "coupon_type": "Cash",
                    "rate": coupon,
                    "freq": {"count": 6, "unit": "months"},
                    "dc": "Act365F",
                    "bdc": "following",
                    "calendar_id": "weekends_only",
                    "stub": "None",
                    "end_of_month": False,
                    "payment_lag_days": 0,
                }
            },
            "discount_curve_id": "USD-OIS",
            "credit_curve_id": None,
            "pricing_overrides": {
                "quoted_clean_price": None, "implied_volatility": None,
                "cds_quote_bp": None, "upfront_payment": None,
                "ytm_bump_decimal": None, "theta_period": None,
                "mc_seed_scenario": None, "adaptive_bumps": False,
                "spot_bump_pct": None, "vol_bump_pct": None,
                "rate_bump_bp": None,
            },
            "call_put": None,
            "attributes": {"tags": [], "meta": {}},
            "settlement_days": None,
            "ex_coupon_days": None,
        },
    }


# Portfolio: 4 bonds with different maturities
PORTFOLIO_BONDS = [
    ("BOND-2Y", 0.045, 2),
    ("BOND-5Y", 0.050, 5),
    ("BOND-7Y", 0.055, 7),
    ("BOND-10Y", 0.060, 10),
]

# Embed the calibrated CreditFactorModel fields directly in the spec.
model_json_str = model.to_json()
model_inline = json.loads(model_json_str)

results = []
for bond_id, coupon, years in PORTFOLIO_BONDS:
    envelope = {
        "schema": "finstack.attribution/1",
        "attribution": {
            "instrument": make_bond_spec(bond_id, coupon, years),
            "market_t0": market_t0,
            "market_t1": market_t1,
            "as_of_t0": AS_OF_T0,
            "as_of_t1": AS_OF_T1,
            "method": "Parallel",
            "config": None,
            "model_params_t0": None,
            "credit_factor_model": model_inline,
            "credit_factor_detail_options": {
                "include_per_bucket_breakdown": True,
                "include_per_issuer_adder": False,
            },
        },
    }
    result_json = attribute_pnl_from_spec(json.dumps(envelope))
    result = json.loads(result_json)
    attr = result["result"]["attribution"]
    results.append({
        "bond": bond_id,
        "total_pnl": float(attr["total_pnl"]["amount"]),
        "carry": float(attr["carry"]["amount"]),
        "rates_curves_pnl": float(attr["rates_curves_pnl"]["amount"]),
        "credit_curves_pnl": float(attr["credit_curves_pnl"]["amount"]),
        "has_credit_detail": attr.get("credit_factor_detail") is not None,
    })

print(f"{'Bond':<10} {'Total P&L':>12} {'Carry':>12} {'Rates':>12} {'Credit':>12} {'CreditDetail?':>14}")
print("-" * 70)
for r in results:
    print(
        f"{r['bond']:<10} {r['total_pnl']:>12.2f} {r['carry']:>12.2f}"
        f" {r['rates_curves_pnl']:>12.2f} {r['credit_curves_pnl']:>12.2f}"
        f" {str(r['has_credit_detail']):>14}"
    )

## Step 7 — Credit vol forecast report

`FactorCovarianceForecast` wraps the calibrated model to produce
horizon-scaled covariance matrices and per-issuer idiosyncratic vol estimates.

In [ ]:
fcf = FactorCovarianceForecast(model)

# One-step annualized covariance
cov_one = json.loads(fcf.covariance_at("one_step"))
factor_ids = cov_one["factor_ids"]
data = np.array(cov_one["data"]).reshape(len(factor_ids), -1)

print("Factor IDs:")
for fid in factor_ids:
    print(f"  {fid}")

print(f"\nCovariance matrix shape: {data.shape}")
print("Diagonal variances (annualized):")
for fid, var in zip(factor_ids, np.diag(data)):
    print(f"  {fid}: {var:.6f}")

# NSteps(12) covariance = 12 × one-step (Sample vol model)
cov_12 = json.loads(fcf.covariance_at('{"n_steps": 12}'))
data_12 = np.array(cov_12["data"]).reshape(len(factor_ids), -1)
ratio = data_12[0, 0] / data[0, 0] if data[0, 0] != 0.0 else float("nan")
print(f"\nNSteps(12) / OneStep ratio (should be 12.0): {ratio:.4f}")

# Per-issuer idiosyncratic vol
print("\nPer-issuer idiosyncratic vol (one-step, annualized):")
for issuer_id in model.issuer_ids():
    vol = fcf.idiosyncratic_vol(issuer_id, "one_step")
    print(f"  {issuer_id}: {vol:.6f}")

## Summary

- `CreditCalibrator` fits a hierarchical spread factor model from a panel of
  issuer spreads and a generic (CDX-like) factor time series.
- `decompose_levels` / `decompose_period` decompose spread moves into generic,
  bucket-level, and issuer-specific (adder) components.
- `attribute_pnl_from_spec` with `credit_factor_model` embedded in the spec
  activates the opt-in credit factor detail in P&L attribution.
- `FactorCovarianceForecast` forecasts factor and idiosyncratic vols at
  arbitrary horizons for risk and scenario analytics.